In [ ]:
# 1. Imports

# Import pandas for data handling
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib for plotting
import matplotlib.pyplot as plt

# Import matplotlib for plot customization
import matplotlib as m

# Import seaborn for statistical visualization
import seaborn as sns

# Import statsmodels for statistical modeling
import statsmodels.api as sm

# Import seasonal_decompose for time-series analysis
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
#testing things

In [ ]:
# 2. Formatting Graph Outputs

# Set Matplotlib to default style for a minimalist look
plt.style.use('default')

# Allow all columns to display without truncation
pd.set_option('display.max_columns', 500)

# Auto-adjust pandas display width for readability
pd.set_option('display.width', None)

# Set default figure size for visible graphs
m.rcParams['figure.figsize'] = (12, 6)

# Set font size for axis labels and tick labels
m.rcParams['axes.labelsize'] = 12
m.rcParams['xtick.labelsize'] = 10
m.rcParams['ytick.labelsize'] = 10

# Make axis titles bold for emphasis
m.rcParams['axes.titleweight'] = 'bold'

# Ensure text color is black for readability
m.rcParams['text.color'] = 'black'

# Increase line width for better visibility
m.rcParams['lines.linewidth'] = 2

# Set axis lines: thin, light gray for subtlety
m.rcParams['axes.axisbelow'] = True
m.rcParams['axes.grid'] = False
m.rcParams['axes.linewidth'] = 0.8
m.rcParams['axes.edgecolor'] = 'gray'

# Add custom grid: light gray, dotted, thin lines
m.rcParams['grid.color'] = 'gray'
m.rcParams['grid.linestyle'] = ':'
m.rcParams['grid.linewidth'] = 0.8
m.rcParams['grid.alpha'] = 0.5

In [ ]:
# 3. Loading the Data
df = pd.read_csv('Electric_Production.csv', skiprows=1, header = None)
#Here we skip the 1st line because it would have showed us the headers of the csv file

# 4. Visualizing the Data
df.head()

In [ ]:
df.columns = ['Month','Energy_Consumption']
df.head()

In [ ]:
df.describe()

In [ ]:
#Month with the last consumption
df.max()

In [ ]:
#Full period of the time series  
print('Start of Time Period: {}'  ' End of Time Period: {}'.format(  
    df.Month.min(), df.Month.max()  
))


In [ ]:
df.shape


In [ ]:
type(df)

In [ ]:
# DATA VISUALIZATION 
df.head


In [ ]:
df.dtypes


In [ ]:
df['Month'] = pd.to_datetime(df['Month'], format='%Y-%m-%d')
df.head()

In [ ]:

#Month as index
df_series = df.set_index('Month')
df_series.head()

In [ ]:
df_series.index

In [ ]:
#Missing data in the time series ? 
df_series.isnull().sum()

In [ ]:
# NICE, if they were we would have used interpolation or polynomiale equivalent IF we hadn't EXACT values for our dataset

In [ ]:
#TIME SERIES TREND ANALYSIS

#Plot the time series
df_series.plot(figsize=(20,12))
plt.show()

In [ ]:
# We observe seasonality and a trend : both are easy to expect since its power consumption
# We can 

In [ ]:
#Density anylysis for E_Consumption
plt.figure(figsize=(15,10))
#Histogram to visualize frequency distribution 
plt.subplot(2,1,1)
df_series['Energy_Consumption'].hist(bins=30,edgecolor='black')
plt.title('Energy_Consumption-Histogram')

#Distribution shape analysis
plt.subplot(2,1,2)
df_series['Energy_Consumption'].plot(kind='kde',linewidth=2,color='darkblue')
plt.title('Energy Consumption - Density Plot')
plt.tight_layout()
plt.show

In [ ]:
# BOX PLOT FOR EACH PERIOD IN THE SERIES 

fig, ax=plt.subplots(figsize=(15,6))
year_index = df_series.index.year
energy_values = df_series['Energy_Consumption']
sns.boxplot(x=year_index, y=energy_values, ax = ax)
plt.xlabel("\nYear")
plt.ylabel("\nEnergy Consumption")
plt.show()


In [ ]:
#TIME SERIE DECOMPOSITION 
#Lets see which decomposition is the best between the additive and the multiplicative one 
decomposition_multiplicative = sm.tsa.seasonal_decompose(df_series,model='multiplicative',extrapolate_trend='freq')
#We are storing the decomposition in an object 


In [ ]:
plt.rcParams.update({'figure.figsize': (16, 10)})

decomposition_multiplicative.plot().suptitle('Multiplicative Decomposition', 
fontsize=22)

plt.show()

In [ ]:
#Original series
#Trend
#Seasonality
#Residuals

#Additive decomposition 
decomposition_additive = sm.tsa.seasonal_decompose(
    df_series, model='additive', extrapolate_trend='freq'
)

type(decomposition_additive)

In [ ]:
plt.figure(figsize=(16, 10))

fig = decomposition_additive.plot()
fig.suptitle('Additive Decomposition', fontsize=22)
plt.show()

In [ ]:
# A test on the correlation and heteroscédasticity of the residual values should be done here 


In [ ]:
#EXtracting time seriess componenents 
df_series_reconstructed = pd.concat(
    [
        decomposition_multiplicative.seasonal,  # Seasonal component
        decomposition_multiplicative.trend,     # Trend component
        decomposition_multiplicative.resid,     # Residual (noise) component
        decomposition_multiplicative.observed   # Observed (original) time series
    ], axis=1
)

In [ ]:
df_series_reconstructed.columns = ['Seasonality', 'Trend', 'Residuals', 'Observed_Values']

df_series_reconstructed.head()

In [ ]:
observed_value =1.111684*3.371515*0.889420
print(observed_value)

In [ ]:
#Statistical Assumptions
#Autocorrelation ? Statioonarity? normal distribution  ? 


# Stationarity ? 
type(df_series)

In [ ]:
rolling_mean = df_series['Energy_Consumption'].rolling(
    window=12).mean()


rolling_std = df_series['Energy_Consumption'].rolling(
    window=12).std()

In [ ]:
# 31. Plot rolling statistics for trend and variance analysis
plt.figure(figsize=(12, 6))

# 31.a Plot original time series values
plt.plot(df_series['Energy_Consumption'], color='blue', label='Original')

# 31.b Plot moving average (rolling mean)
plt.plot(rolling_mean, color='red', label='Moving Average')

# 31.c Plot rolling standard deviation
plt.plot(rolling_std, color='black', label='Standard Deviation')

# 31.d Configure legend and title
plt.legend(loc='best')
plt.title('Rolling Statistics - Moving Average & Standard Deviation')
plt.show()

In [ ]:
#Autocorrelation check 
# ACF = AUtocorrelation function 
#PACF = Partial Autocorrelation Function 
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams.update({'figure.figsize': (16, 10)})

plt.subplot(2, 1, 1)

plot_acf(df_series['Energy_Consumption'], 
         ax=plt.gca(), lags=30)

plt.subplot(2, 1, 2)

plot_pacf(df_series['Energy_Consumption'], 
          ax=plt.gca(), lags=30, method='ywm')

plt.show()

In [ ]:
#we observe BIG autocorrelation 

In [ ]:
# Stationary test : Dickey-Fuller test
from statsmodels.tsa.stattools import adfuller


print('\\nDickey-Fuller Test Results:\\n')
adf_test = adfuller(df_series['Energy_Consumption'], 
                     autolag='AIC')

df_output = pd.Series(adf_test[0:4], index=[
    'Test Statistic', 'p-value', 
    'Lags Used', 'Observations Used'])

for key, value in adf_test[4].items():
    df_output[f'Critical Value ({key})'] = value

print(df_output)

In [ ]:
#p-value of 0,88 dff of 0,05 
# We can also automate the test : 


In [ ]:
#Log-transformation 
df_series['Energy_Consumption_Log'] = np.log(
    df_series['Energy_Consumption'])

df_series.head()

In [ ]:
plt.plot(df_series['Energy_Consumption_Log'], 
         color="blue")


In [ ]:
plt.subplot (2,1,1)
plt.hist(df_series['Energy_Consumption_Log'],color="blue")

In [ ]:
def test_stationarity(series):

    #Import test
    from statsmodels.tsa.stattools import adfuller
    
    rolling_mean = series.rolling(12).mean()
    rolling_std = series.rolling(12).std()
    
    #rolling statistics
    plt.figure(figsize=(12, 6))
    plt.plot(series, color='blue', label='Original')
    plt.plot(rolling_mean, color='red', label='Moving Avg')
    plt.plot(rolling_std, color='black', label='Std Dev')
    plt.legend(loc='best')
    plt.title('Rolling Statistics')
    plt.show()
    
    #Dickey-Fuller test
    print("\nDickey-Fuller Test Results:\n")
    adf_test = adfuller(series, autolag='AIC')
    df_output = pd.Series(adf_test[0:4], index=[
        'Test Statistic', 'p-value', 
        'Lags Used', 'Observations Used'])
    
    #critical values
    for key, value in adf_test[4].items():
        df_output[f'Critical Value ({key})'] = value
    print(df_output)
    
    print("\nConclusion:")
    if df_output['p-value'] > 0.05:
        print("\nThe p-value is > 0.05, failing to reject H₀.")
        print("The series is likely non-stationary.")
    else:
        print("\nThe p-value is < 0.05, rejecting H₀.")
        print("The series is likely stationary.")



In [ ]:
test_stationarity(df_series['Energy_Consumption_Log'])

In [ ]:
# SQUARE ROOT TRANSFO


In [ ]:
test_stationarity(df_series['Energy_Consumption_Log'])